In [2]:
# ------------------------------------------------------------------------------
# Setup and Path Validation
# ------------------------------------------------------------------------------
from app.core.tools.catalog_tool import catalog_lookup, CATALOG_PATH
import os
import json
from pathlib import Path
import sys

# Ensure we can import catalog_tool
HERE = Path.cwd().resolve()
print("Running from:", HERE)


def is_repo_root(p: Path) -> bool:
    # Prefer strong layout markers to avoid accidentally selecting /scripts as root
    return (
        (p / "pyproject.toml").exists()
        or (p / ".git").exists()
        or ((p / "app").exists() and (p / "scripts").exists() and (p / "data").exists())
        or ((p / "knowledge").exists() and (p / "data").exists())
        or (p / "main.py").exists()
    )

# Adjust this to point to your repo root so 'data/catalog.json' is reachable
PROJECT_ROOT = next(p for p in [HERE, *HERE.parents] if is_repo_root(p))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Catalog Path: {CATALOG_PATH}")
print(f"Catalog exists? {CATALOG_PATH.exists()}")

Running from: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/tests
Project Root: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot
Catalog Path: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/data/catalog.json
Catalog exists? True


In [3]:
# ------------------------------------------------------------------------------
# Inspect Catalog Data
# ------------------------------------------------------------------------------
# Let's see what we are working with
if CATALOG_PATH.exists():
    data = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
    # Note: your ingest script uses "items", tool uses "products"
    items = data.get("items", [])
    print(f"Total items in catalog: {len(items)}")

    # Check first 2 items to see structure
    for item in items[:2]:
        print(
            f"- ID: {item['id']} | Title: {item['title']} | Family: {item['family']}")
else:
    print("❌ ERROR: catalog.json not found. Run ingest_raw_catalog.ipynb first.")


Total items in catalog: 247
- ID: 205.112 | Title: Comando Connect - Repuesto De Climatizador Neil | Family: CLIMATIZADOR
- ID: 3216.05.2 | Title: Goma de Sensor - Repuesto Climatizadores  Neil | Family: CLIMATIZADOR


In [4]:
# ------------------------------------------------------------------------------
#  Helper Function for Pretty Printing Results
# ------------------------------------------------------------------------------

def test_query(query, family=None, k=3):
    print(f"\nSearching for: '{query}' | Family Filter: {family}")
    try:
        results = catalog_lookup(query, product_family=family, k=k)
        print(f"Found {results['count']} matches (showing top {k}):")
        for i, match in enumerate(results['matches'], 1):
            price = match.get('price', 'N/A')
            print(
                f"  {i}. [{match.get('id')}] {match.get('title')} - ${price} {results['currency']}")
    except Exception as e:
        print(f"❌ Search failed: {e}")

In [5]:
# ------------------------------------------------------------------------------
# 5) Functional Smoke Tests
# ------------------------------------------------------------------------------


# Test 1: Exact ID lookup
# Replace 'ID_HERE' with a real ID from your catalog.json
test_query("C101")

# Test 2: Title lookup (Fuzzy/Token match)
test_query("HDK 2200")

# Test 3: Family filtering (Should only return TPMS)
test_query("sensor", family="TPMS")

# Test 4: Broad search (Should return multiple items up to k=5)
test_query("aire", k=5)


Searching for: 'C101' | Family Filter: None
Found 3 matches (showing top 3):
  1. [C035] TPMS con conexión a encendedor + Puerto USB - Sensores externos - C101 - $78159.95 ARS
  2. [C036] TPMS con conexión a encendedor + Puerto USB - Sensores internos - C101 - $88056.54 ARS
  3. [TP009] Sensor Externo Para Tpms Autos Y Camionetas C260 Y C410 - $18336.0 ARS

Searching for: 'HDK 2200' | Family Filter: None
Found 3 matches (showing top 3):
  1. [C04312_E] Aire Acondicionado 12V para Camiones - Inverter 2200W - NEIL HDK 2200 - $1822013.0 ARS
  2. [C04324] Aire Acondicionado 24V para Camion - Inverter 2200W NEIL HDK 2200 - $1822013.0 ARS
  3. [R00036] Compresor + Controlador (12V) -Repuesto AA HDK 2200- - $882458.0 ARS

Searching for: 'sensor' | Family Filter: TPMS
Found 3 matches (showing top 3):
  1. [C019] TPMS para Utilitarios y Motorhome - Sensores Internos - C300 - $188301.0 ARS
  2. [C022] Tpms para autos - Sensores Externos - C260 - $77616.67 ARS
  3. [C022+1] TPMS para Autos - Sen

In [6]:
# ------------------------------------------------------------------------------
# 6) Edge Case Testing
# ------------------------------------------------------------------------------

print("\n--- Edge Case Testing ---")

# Test: Case Insensitivity
test_query("tpms c101")
test_query("TPMS")

# Test: Non-existent product
test_query("Flux Capacitor 9000")

# Test: Empty Query (behavior check)
test_query("")


--- Edge Case Testing ---

Searching for: 'tpms c101' | Family Filter: None
Found 3 matches (showing top 3):
  1. [C035] TPMS con conexión a encendedor + Puerto USB - Sensores externos - C101 - $78159.95 ARS
  2. [TP029] Monitor Tpms C101 + Usb  -neil- Solo Repuesto - $25997.0 ARS
  3. [C036] TPMS con conexión a encendedor + Puerto USB - Sensores internos - C101 - $88056.54 ARS

Searching for: 'TPMS' | Family Filter: None
Found 3 matches (showing top 3):
  1. [C019] TPMS para Utilitarios y Motorhome - Sensores Internos - C300 - $188301.0 ARS
  2. [C022] Tpms para autos - Sensores Externos - C260 - $77616.67 ARS
  3. [C022+1] TPMS para Autos - Sensores Externos + Auxilio - C260+1 - $89258.6 ARS

Searching for: 'Flux Capacitor 9000' | Family Filter: None
Found 0 matches (showing top 3):

Searching for: '' | Family Filter: None
Found 3 matches (showing top 3):
  1. [205.112] Comando Connect - Repuesto De Climatizador Neil - $143847.0 ARS
  2. [3216.05.2] Goma de Sensor - Repuesto Climati